In [1]:
import pickle 
import numpy as np
import re

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector, Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import CXGate, PauliEvolutionGate
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router import CommutingGateRouter
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph
from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian

In [2]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [3]:
hubo_hams = {}
hubo_circs = {}
hubo_reordered_hams = {}
results = {}
for name, filename, copy_numbers in zip(
    [
        # 'H_6', 'H_9', 'H_12', 'H_20(1)',
        'H_12', 'H_15', 'H_16', 'H_20(1)',
    # 'H_20(2)', 'H_30'
    ],
    [
        # 'test_N2_W2', 'trivial', 'test_N3_W4', 'test_N4_W5', 
        'test_N3_W4', 'test_N3_W5', 'test_N7_W4', 'test_N4_W5', 
        # 'test_N7_W5', 'test_N10_W6'
    ], 
    [
        # [1,1], [1,1,1], [2,1,1], [2,1,1,1], 
        [2,1,1],[2,2,1], [1,1,1,0,0,0,1], [2,1,1,1], 
        # [1,1,1,0,1,0,1], [1,1,0,0,1,0,1,1,0,1]
    ]
):
    filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
    graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)

    full_hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
    hubo_hams[name] = full_hamiltonian
    
    ess = ExtendedSwapStrategy.from_all_to_all(n * T)
    num_physical_qubits = ess._num_vertices
    
    pm_rz = PassManager(
        [
            FindCommutingPauliEvolutionsMulti(), 
            CommutingGateRouter(
                ess,
                max_layers=0,
                perform_extra_swaps=True
            ),
            SwapToFinalMapping(),
            InverseCancellation(gates_to_cancel=[CXGate()]),
            CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
            InverseCancellation(gates_to_cancel=[CXGate()]),
        ]
    )
    qc = QuantumCircuit(num_physical_qubits)
    qc.append(PauliEvolutionGate(full_hamiltonian, time=Parameter('γ')), range(num_physical_qubits))   
    tqc_rz = pm_rz.run(qc)
    print_circuit_info(tqc_rz, 'Rz circuit')
    hubo_circs[name] = tqc_rz
    
    # reordered_hamiltonian = SparsePauliOp.from_sparse_list([], tqc_rz.num_qubits)
    # for instruction in tqc_rz.data:
    #     if instruction.operation.name == 'rz':
    #         qubits_str = str(instruction.qubits)
    #         matches = re.findall('index=([0-9]+)', qubits_str)
    #         if not len(matches) == 1:
    #         raise Exception('Bad Rz')
    #         reordered_hamiltonian += SparsePauliOp.from_sparse_list([('Z', [int(matches[0])], instruction.operation.params[0] / 2)], tqc_rz.num_qubits)
    #     if instruction.operation.name == 'PauliEvolution':
    #         qubits_str = str(instruction.qubits)
    #         matches = re.findall('index=([0-9]+)', qubits_str)
    #         reordered_hamiltonian += SparsePauliOp.from_sparse_list([('Z' * len(matches), [int(x) for x in matches], instruction.operation.params[0])], tqc_rz.num_qubits)
    # hubo_reordered_hams[name] = reordered_hamiltonian


Keeping constraints at times: [0 1 2]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 12 qubits
Rz circuit has 228 non-local gates and 120 non-local depth
Rz circuit contains ['cx', 'rz'] gates.
Keeping constraints at times: [1 2 0 3]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 15 qubits
Rz circuit has 330 non-local gates and 136 non-local depth
Rz circuit contains ['cx', 'rz'] gates.
Keeping constraints at times: [2 1 0]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 16 qubits
Rz circuit has 1028 non-local gates and 531 non-local depth
Rz circuit contains ['cx', 'rz'] gates.
Keeping constraints at times: [1 3 0 2]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 20 qubits
Rz cir

In [4]:
hubo_circs.keys()

dict_keys(['H_12', 'H_15', 'H_16', 'H_20(1)'])

In [8]:
for i in range(1, 4):
    with open(f'/lustre/scratch127/qpg/jc59/out/qiskit/hubo_no_shot_noise_optimum/sweep.test_N3_W4.p{i}.pkl', 'rb') as f:
        res = pickle.load(f)
    print(res['best_func_val'], res['best_params'])

-0.013284094482182165 [0.48647931 2.95548637]
-0.026564533655324126 [0.53370861 0.23797497 2.94663149 0.46243552]
-0.035531350903866735 [0.08281156 0.4930252  0.26473587 0.83069666 2.1462015  2.42471994]


In [7]:
for i in range(1, 4):
    with open(f'/lustre/scratch127/qpg/jc59/out/qiskit/hubo_no_shot_noise_optimum/sweep.test_N3_W5.p{i}.pkl', 'rb') as f:
        res = pickle.load(f)
    print(res['best_func_val'], res['best_params'])

-0.0016678453802346444 [0.42052729 2.9814083 ]
-0.004392185610614987 [2.84819588 2.87340989 0.68907573 0.73433223]
-0.006453453360317543 [ 0.58873781  0.47460558  2.87431393  0.0960188  -0.08696553  0.06875723]


In [9]:
for i in range(1, 4):
    with open(f'/lustre/scratch127/qpg/jc59/out/qiskit/hubo_no_shot_noise_optimum/sweep.test_N7_W4.p{i}.pkl', 'rb') as f:
        res = pickle.load(f)
    print(res['best_func_val'], res['best_params'])

-0.0012275349865603323 [ 2.7325505  -2.38950953]
-0.0069367109816559115 [ 2.77945932  2.83781335 -2.97713163 -2.44348794]
-0.009629663908613278 [ 2.84846336  2.77242027  2.88426847  0.116586   -2.42611002  0.15472621]


In [10]:
for i in range(1, 4):
    with open(f'/lustre/scratch127/qpg/jc59/out/qiskit/hubo_no_shot_noise_optimum/sweep.test_N4_W5.p{i}.pkl', 'rb') as f:
        res = pickle.load(f)
    print(res['best_func_val'], res['best_params'])

-0.0001936683577346888 [ 2.7126842  -2.97022108]
-0.0014170797958546403 [ 0.51896757  0.30245932  2.99314664 -0.14044104]
-0.00336431884256319 [ 2.07036248  0.57071544  2.83268046 -0.2156792  -0.11957621  0.10250635]


In [11]:
all_params = {
    'test_N2_W2': {
        1: [np.float64(2.623573572855404), np.float64(-2.950436618484883)],
        2: [np.float64(2.2786551178186687),np.float64(0.3484333612969359),np.float64(-0.20958461281251378), np.float64(-0.18035666937489597)],
        3: [np.float64(2.103494296555067),np.float64(2.331964628588021),np.float64(2.766077743539691),np.float64(-2.3298722370413767),np.float64(0.9304533389315034),np.float64(-2.3822537184722923)]
    },
    'trivial': {
        1: [np.float64(0.4604817138832364),np.float64(2.934248722163311)],
        2: [np.float64(2.7973874712118962),np.float64(0.2705232833930648), np.float64(-2.9358911658286457),np.float64(2.9068223766900205)],
        3: [np.float64(0.40121342948119815),np.float64(0.33932426908737173),np.float64(2.825988092432951),  np.float64(2.427887017206748),np.float64(2.490507928475604),np.float64(2.6738386528394535)]
    },
    'test_N3_W4': {
        1: [np.float64(0.4864793104929956), np.float64(2.9554863744976756)],
        2: [np.float64(0.5337086148294259),np.float64(0.23797497165829454),np.float64(2.9466314912565448),np.float64(0.4624355197846208)],
        3: [np.float64(0.08281155900313397),np.float64(0.4930252006439344),np.float64(0.264735874038035),np.float64(0.8306966597816092),np.float64(2.1462015043805587),np.float64(2.4247199409271656)]
    },
    'test_N4_W5': {
        1: [np.float64(2.7126842047317505),np.float64(-2.970221081389355)],
        2: [np.float64(0.5189675671501274),np.float64(0.30245932207616016),np.float64(2.99314664253983),np.float64(-0.14044104492507986)],
        3: [np.float64(2.0703624839394834),np.float64(0.5707154409650862),np.float64(2.8326804591568164),np.float64(-0.21567919947745579),np.float64(-0.11957620650026374),np.float64(0.10250635166105089)]
    },
    'test_N3_W5': {
        1: [0.42052729, 2.9814083],
        2: [2.84819588, 2.87340989, 0.68907573, 0.73433223],
        3: [ 0.58873781,  0.47460558,  2.87431393,  0.0960188,  -0.08696553,  0.06875723]
    },
    'test_N7_W4': {
        1: [ 2.7126842,  -2.97022108],
        2: [ 0.51896757,  0.30245932,  2.99314664, -0.14044104],
        3: [ 2.07036248,  0.57071544,  2.83268046, -0.2156792,  -0.11957621,  0.10250635]
    }
}

all_optimised_circuits = {}

for filename, name in zip(
    ['test_N3_W5', 'test_N7_W4',],
    ['H_15', 'H_16']
):
    optimised_circuits = {}
    cost_circuit = hubo_circs[name]
    num_qubits = cost_circuit.num_qubits
    for p in range(1, 4):
        gammas = ParameterVector("γ", p)
        betas = ParameterVector("β", p)
        
        qaoa_circuit = QuantumCircuit(num_qubits, num_qubits)
        qaoa_circuit.h(range(num_qubits))
        
        
        mixer_layer = QuantumCircuit(num_qubits)
        beta = Parameter("β")
        mixer_layer.rx(2 * beta, range(num_qubits))

        
        for layer in range(p):
            bind_dict = {cost_circuit.parameters[0]: all_params[filename][p][p+layer]}
            bound_cost_layer = cost_circuit.assign_parameters(bind_dict)

            bind_dict = {mixer_layer.parameters[0]: all_params[filename][p][layer]}
            bound_mixer_layer = mixer_layer.assign_parameters(bind_dict)

            qaoa_circuit.compose(bound_cost_layer, range(num_qubits), inplace=True)
            qaoa_circuit.compose(bound_mixer_layer, range(num_qubits), inplace=True)
        
        for idx in range(num_qubits):
            qaoa_circuit.measure(idx, idx)
            
        
        optimised_circuits[p] = qaoa_circuit
            
    all_optimised_circuits[name] = optimised_circuits

In [14]:
all_optimised_circuits['H_15'][1].draw(fold=-1)

global phase: 4.0602
      ┌───┐ ┌────────────┐ ┌───┐┌─────────────┐┌───┐┌────────────┐┌───┐┌────────────┐┌───┐┌────────────┐┌───┐┌────────────┐┌───┐┌─────────────┐┌───┐┌─────────────┐┌───┐┌─────────────┐┌───┐┌────────────┐┌───┐┌────────────┐┌───┐┌─────────────┐┌───┐┌────────────┐┌───┐┌────────────┐┌───┐┌────────────┐┌───┐┌─────────────┐┌───┐     ┌───┐┌───┐┌───┐ ┌────────────┐     ┌───┐     ┌─────────────┐    ┌───┐     ┌─────────────┐    ┌───┐     ┌─────────────┐    ┌───┐     ┌─────────────┐    ┌───┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                      ┌───┐┌────────────┐┌───┐┌─────────────┐          ┌───┐     ┌──────────────┐               ┌───┐                                                                              ┌───┐┌────────────┐              ┌───┐┌─────────────┐                  ┌───┐     ┌──────────────┐          ┌───┐              ┌───┐ ┌────────────┐     ┌───┐┌─────────────┐               ┌───┐┌──────────────┐               ┌───┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    ┌─────────────┐                        ┌─┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
 q_0: ┤ H ├─┤ Rz(8.5715) ├─┤ X ├┤ Rz(-14.534) ├┤ X ├┤ Rz(2.6087) ├┤ X ├┤ Rz(3.7268) ├┤ X ├┤ Rz(3.7268) ├┤ X ├┤ Rz(1.8634) ├┤ X ├┤ Rz(-1.8634) ├┤ X ├┤ Rz(-3.7268) ├┤ X ├┤ Rz(-3.7268) ├┤ X ├┤ Rz(4.0994) ├┤ X ├┤ Rz(2.6087) ├┤ X ├┤ Rz(-2.6087) ├┤ X ├┤ Rz(2.6087) ├┤ X ├┤ Rz(1.8634) ├┤ X ├┤ Rz(1.8634) ├┤ X ├┤ Rz(-3.7268) ├┤ X ├─────┤ X ├┤ X ├┤ X ├─┤ Rz(4.0994) ├─────┤ X ├─────┤ Rz(-5.5901) ├────┤ X ├─────┤ Rz(-1.8634) ├────┤ X ├─────┤ Rz(-1.8634) ├────┤ X ├─────┤ Rz(-1.8634) ├────┤ X ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■───────────────────────────■───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(2.2361) ├┤ X ├┤ Rz(0.74535) ├

In [15]:
from qiskit.qasm3 import dump as qasm3_dump
for name in all_optimised_circuits.keys():
    for p in range(1, 4):
        with open(f'../out/ionq/qasm3/optimised_hubo_circuit_{name}_{p}.txt', 'w') as f:
            qasm3_dump(all_optimised_circuits[name][p], f)

In [ ]:
# from qiskit.qasm3 import dump as qasm3_dump
# for name in hubo_circs.keys():
#     with open(f'../out/ionq/qasm3/hubo_circuit_{name}.txt', 'w') as f:
#         qasm3_dump(hubo_circs[name], f)

In [ ]:
# from qiskit import qasm2
# for name in hubo_circs.keys():
#     with open(f'../out/ionq/qasm2/hubo_circuit_{name}.txt', 'w') as f:
#         qasm2.dump(hubo_circs[name], f)

In [ ]:
# with open('../out/ionq/hubo_hamiltonians.pkl', 'wb') as f:
#     pickle.dump(hubo_hams, f)

In [ ]:
# with open('../out/ionq/hubo_reordered_hamiltonians.pkl', 'wb') as f:
#     pickle.dump(hubo_reordered_hams, f)

In [ ]:
# from qiskit.qasm3 import load
# hubo_circs = dict()
# for name in     [
#         'H_6', 'H_9', 'H_12', 
#         'H_20(1)', 'H_20(2)', 'H_30'
#     ]:
#     hubo_circs[name] = load(f'../out/ionq/qasm3/hubo_circuit_{name}.txt')
#     # with open(f'../out/ionq/qasm3/hubo_circuit_{name}.txt', 'r') as f:
        

In [ ]:
# hubo_circs['H_6'].draw(fold=-1)

In [ ]:
# from qiskit.qasm2 import load
# hubo_circs = dict()
# for name in     [
#         'H_6', 'H_9', 'H_12', 
#         'H_20(1)', 'H_20(2)', 'H_30'
#     ]:
#     hubo_circs[name] = load(f'../out/ionq/qasm2/hubo_circuit_{name}.txt')
#     # with open(f'../out/ionq/qasm3/hubo_circuit_{name}.txt', 'r') as f:

In [33]:
from qiskit_aer import AerSimulator


In [34]:
ess = ExtendedSwapStrategy.from_all_to_all(3 *2)
backend = AerSimulator(coupling_map=ess._coupling_map,)

In [36]:
qc = all_optimised_circuits['H_6'][1]
qc.draw(fold=-1)

global phase: 4.9518
     ┌───┐┌─────────────┐                                                                                                                                                                                                            ┌───┐┌────────────┐                         ┌───┐┌─────────────┐┌───┐┌─────────────┐┌───┐               ┌───┐┌───────────┐               ┌───┐                                                                                                                                                                                                                                               ┌────────────┐              ┌─┐                       
q_0: ┤ H ├┤ Rz(-26.185) ├────────────────■────────────────────────────────────────────────────────■─────────────────────────────────────■────────────────────────────────────────────────────────■───────────────────────────────────┤ X ├┤ Rz(0.3688) ├─────────────────────────┤ X ├┤ Rz(-3.3192) ├┤ X ├┤ Rz(-3.3192) ├┤ X ├───────────────┤ X ├┤ Rz(1.844) ├───────────────┤ X ├───────────────────────────────────■───────────────────────────────────■──────────────────────────────────────────────■───────────────────────────────────────■───────────────────────────────────────────────────────────■─────────────────■──┤ Rx(5.2471) ├──────────────┤M├───────────────────────
     ├───┤└────┬───┬────┘┌────────────┐┌─┴─┐┌────────────┐┌───┐┌────────────┐┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐┌────────────┐┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐┌────────────┐└─┬─┘└───┬───┬────┘          ┌───┐┌───┐┌───┐└─┬─┘└─────────────┘└─┬─┘└─────────────┘└─┬─┘┌─────────────┐└─┬─┘└───┬───┬───┘┌─────────────┐└─┬─┘┌───┐     ┌───┐                    │                                   │                    ┌───┐┌────────────┐┌───┐  │  ┌────────────┐     ┌─┐               │                                                           │                 │  └────────────┘              └╥┘                       
q_1: ┤ H ├─────┤ X ├─────┤ Rz(-1.844) ├┤ X ├┤ Rz(-1.844) ├┤ X ├┤ Rz(-1.844) ├┤ X ├┤ Rz(-1.844) ├┤ X ├┤ Rz(-1.844) ├┤ X ├┤ Rz(-1.844) ├┤ X ├┤ Rz(-1.844) ├┤ X ├┤ Rz(-1.844) ├┤ X ├┤ Rz(-1.844) ├┤ X ├┤ Rz(-1.844) ├┤ X ├┤ Rz(-1.844) ├──┼──────┤ X ├───────────────┤ X ├┤ X ├┤ X ├──■───────────────────┼───────────────────■──┤ Rz(-3.3192) ├──┼──────┤ X ├────┤ Rz(-3.3192) ├──┼──┤ X ├─────┤ X ├────────────────────┼───────────────────────────────────┼────────────────────┤ X ├┤ Rz(-1.844) ├┤ X ├──┼──┤ Rx(5.2471) ├─────┤M├───────────────┼───────────────────────────────────────────────────────────┼─────────────────┼───────────────────────────────╫────────────────────────
     ├───┤     └─┬─┘     └────────────┘└───┘└────────────┘└─┬─┘└────────────┘└─┬─┘└────────────┘├───┤└────────────┘└─┬─┘└────────────┘└───┘└────────────┘└─┬─┘└────────────┘└─┬─┘└────────────┘└───┘└────────────┘└─┬─┘└────────────┘  │      └─┬─┘     ┌───┐     └─┬─┘└─┬─┘└─┬─┘┌───┐ ┌───────────┐   │                      └─────────────┘  │      └─┬─┘    └─────────────┘  │  └─┬─┘┌───┐└─┬─┘┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐└─┬─┘└────────────┘└─┬─┘  │  └────────────┘     └╥┘               │                    ┌───┐         ┌───┐     ┌───────────┐┌─┴─┐┌───────────┐┌─┴─┐    ┌───┐     ┌────────────┐ ║               ┌─┐      
q_2: ┤ H ├───────■──────────────────────────────────────────┼──────────────────┼────────────────┤ X ├────────────────┼─────────────────────────────────────■──────────────────┼─────────────────────────────────────┼──────────────────┼────────■───────┤ X ├───────┼────■────┼──┤ X ├─┤ Rz(1.844) ├───┼───────────────────────────────────────■────────┼───────────────────────■────┼──┤ X ├──┼──┤ X ├┤ Rz(1.844) ├┤ X ├┤ Rz(1.844) ├┤ X ├┤ Rz(1.844) ├┤ X ├┤ Rz(1.844) ├┤ X ├──■──────────────────■────┼──────────────────────╫────────────────┼────────────────────┤ X ├─────────┤ X ├─────┤ Rz(1.844) ├┤ X ├┤ Rz(1.844) ├┤ X ├────┤ X ├─────┤ Rx(5.2471) ├─╫──────────────

In [ ]:
# qc = QuantumCircuit(hubo_circs['H_6'].num_qubits)
# qc.h(range(qc.num_qubits))
# qc.compose(hubo_circs['H_6'], inplace=True)
# qc.measure_all()
# qc.draw(fold=-1)


In [37]:
qc.count_ops()

OrderedDict([('cx', 52), ('rz', 31), ('h', 6), ('rx', 6), ('measure', 6)])

In [38]:
res = backend.run(qc).result()

In [41]:
from collections import Counter
counts = Counter(res.get_counts())
counts.most_common(10)

[('000010', 74),
 ('111100', 63),
 ('110100', 59),
 ('001010', 55),
 ('001000', 43),
 ('111000', 42),
 ('001110', 39),
 ('101010', 37),
 ('011010', 37),
 ('111110', 36)]

In [ ]:
# with open('../out/ionq/hubo_reordered_hamiltonians.pkl', 'rb') as f:
#     reorder_hams_check= pickle.load(f)

In [44]:
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples
evaluate_sparse_pauli_samples([x[0] for x in counts.most_common(20)], hubo_hams['H_6'])

array([ 0., 11.,  0., 11., 11., 11., 11., 11., 11., 11., 10., 11., 11.,
       10., 11., 10., 10., 11., 12., 12.])